In [ ]:
"""
Content Pipeline Module
Implements: Document → Monitor → Brief → Publish → Iterate
"""

from datetime import datetime
from typing import Any, Callable

from src.knowledge_base import KnowledgeBase
from src.prompt_templates import PromptTemplates
from src.llm_integration import LLMIntegration


class ContentPipeline:
    """Full content creation pipeline: Document → Monitor → Brief → Publish → Iterate."""

    # -- Brand constants (single source of truth) --
    BRAND_VOICE = "Friendly, practical, optimistic, approachable"
    TARGET_AUDIENCE = "Eco-conscious homeowners aged 25-45"
    BRAND_GOAL = "Educate and empower while naturally positioning GreenPulse products"

    KEY_MESSAGES = [
        "Sustainability is achievable with small steps",
        "Smart home tech saves money AND helps the planet",
        "GreenPulse makes it easy to get started",
    ]

    AVOID = [
        "Generic marketing clichés",
        "Fear-based messaging",
        "Overly technical jargon",
        "Aggressive sales language",
    ]

    MARKET_INSIGHTS = {
        "content_gap": "Educational content about energy savings",
        "market_trend": "Consumers want sustainability + convenience",
        "competitor_weakness": "Generic, salesy, or fear-based content",
    }

    def __init__(self, primary_path: str, secondary_path: str):
        self.kb = KnowledgeBase(primary_path, secondary_path)
        self.llm = LLMIntegration()
        self.templates = PromptTemplates()
        self.pipeline_log: list[dict] = []

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------
    def _log(self, stage: str, message: str) -> None:
        """Record and print a pipeline activity entry."""
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "stage": stage,
            "message": message,
        }
        self.pipeline_log.append(entry)
        print(f"[{entry['timestamp']}] {stage}: {message}")

    @staticmethod
    def _print_header(title: str) -> None:
        bar = "=" * 60
        print(f"\n{bar}\n  {title}\n{bar}")

    @staticmethod
    def _print_section(title: str) -> None:
        print(f"\n--- {title} ---")

    def _get_template_for(self, content_type: str) -> Callable:
        """Map a content type to its template builder. Defaults to blog post."""
        dispatch = {
            "blog post": lambda topic, p, s: self.templates.blog_post(topic, p, s),
            "social media": lambda topic, p, s: self.templates.social_media(topic, "Instagram", p, s),
            "email newsletter": lambda topic, p, s: self.templates.email_newsletter(topic, p, s),
            "product description": lambda topic, p, s: self.templates.product_description(topic, p, s),
        }
        return dispatch.get(content_type, dispatch["blog post"])

    # ------------------------------------------------------------------
    # Stage 1: Document
    # ------------------------------------------------------------------
    def document_stage(self) -> dict:
        """Ingest and process documents from both knowledge bases."""
        self._log("DOCUMENT", "Loading knowledge bases...")
        result = self.kb.load_all()
        self._log(
            "DOCUMENT",
            f"Loaded {result['primary_count']} primary, "
            f"{result['secondary_count']} secondary docs",
        )
        self._log("DOCUMENT", "Document ingestion complete")
        return self.kb.get_summary()

    # ------------------------------------------------------------------
    # Stage 2: Monitor
    # ------------------------------------------------------------------
    def monitor_stage(self, focus_keyword: str | None = None) -> dict:
        """Analyze content needs and market alignment."""
        self._log("MONITOR", "Analyzing content landscape...")

        analysis: dict[str, Any] = {
            "brand_voice": self.BRAND_VOICE,
            "target_audience": self.TARGET_AUDIENCE,
            **self.MARKET_INSIGHTS,
        }

        if focus_keyword:
            analysis["keyword_search"] = self.kb.search(focus_keyword)
            self._log("MONITOR", f"Found mentions of '{focus_keyword}' in knowledge base")

        self._log("MONITOR", "Monitoring analysis complete")
        return analysis

    # ------------------------------------------------------------------
    # Stage 3: Brief
    # ------------------------------------------------------------------
    def brief_stage(self, content_type: str, topic: str) -> dict:
        """Generate a content brief based on knowledge base insights."""
        self._log("BRIEF", f"Creating brief for {content_type}: {topic}")

        brief = {
            "content_type": content_type,
            "topic": topic,
            "audience": self.TARGET_AUDIENCE,
            "brand_voice": self.BRAND_VOICE,
            "goal": self.BRAND_GOAL,
            "key_messages": self.KEY_MESSAGES,
            "avoid": self.AVOID,
        }

        self._log("BRIEF", "Content brief created")
        return brief

    # ------------------------------------------------------------------
    # Stage 4: Publish
    # ------------------------------------------------------------------
    def publish_stage(self, content_type: str, topic: str) -> dict:
        """Generate final content using the LLM with knowledge base context."""
        self._log("PUBLISH", f"Generating {content_type} about: {topic}")

        primary_ctx = self.kb.get_primary_context()
        secondary_ctx = self.kb.get_secondary_context()

        build_prompt = self._get_template_for(content_type)
        prompt = build_prompt(topic, primary_ctx, secondary_ctx)

        result = self.llm.generate(prompt)
        self._log("PUBLISH", f"Content generated via {result['provider']}")
        return result

    # ------------------------------------------------------------------
    # Stage 5: Iterate
    # ------------------------------------------------------------------
    def iterate_stage(self, content: str, feedback: str = "") -> dict:
        """Refine content based on feedback."""
        self._log("ITERATE", "Refining content...")
        prompt = self.templates.content_refinement(content, feedback)
        result = self.llm.generate(prompt)
        self._log("ITERATE", f"Content refined via {result['provider']}")
        return result

    # ------------------------------------------------------------------
    # Full pipeline
    # ------------------------------------------------------------------
    def run_full_pipeline(self, content_type: str, topic: str) -> dict:
        """Run the complete pipeline end-to-end."""
        self._print_header("GREENPULSE CONTENT PIPELINE")

        self._print_section("STAGE 1: DOCUMENT")
        doc_summary = self.document_stage()

        self._print_section("STAGE 2: MONITOR")
        monitoring = self.monitor_stage(topic.split()[0] if topic else None)

        self._print_section("STAGE 3: BRIEF")
        brief = self.brief_stage(content_type, topic)
        for key, value in brief.items():
            print(f"  {key}: {value}")

        self._print_section("STAGE 4: PUBLISH")
        published = self.publish_stage(content_type, topic)
        print(f"\n{published['content']}")

        self._print_section("STAGE 5: ITERATE")
        refined = self.iterate_stage(published["content"])
        print(f"\n{refined['content']}")

        self._print_header("PIPELINE COMPLETE")

        return {
            "document_summary": doc_summary,
            "monitoring": monitoring,
            "brief": brief,
            "published_content": published,
            "refined_content": refined,
            "pipeline_log": self.pipeline_log,
        }

    # ------------------------------------------------------------------
    # Uniqueness comparison
    # ------------------------------------------------------------------
    def compare_with_generic(self, content_type: str, topic: str) -> dict:
        """Generate both brand-aligned and generic content for side-by-side comparison."""
        self._print_header("UNIQUENESS COMPARISON")

        self._print_section("BRAND-ALIGNED OUTPUT")
        brand_result = self.publish_stage(content_type, topic)
        print(brand_result["content"])

        self._print_section("GENERIC AI OUTPUT")
        generic_prompt = self.templates.generic_prompt(content_type, topic)
        generic_result = self.llm.generate(generic_prompt)
        print(generic_result["content"])

        self._print_section("DIFFERENCES")
        differences = [
            "Brand output uses GreenPulse-specific products and voice",
            "Brand output avoids generic marketing clichés",
            "Brand output reflects market insights from knowledge base",
            "Generic output is broad and could apply to any company",
        ]
        for i, diff in enumerate(differences, start=1):
            print(f"{i}. {diff}")

        return {
            "brand_content": brand_result,
            "generic_content": generic_result,
        }

ModuleNotFoundError: No module named 'src'